In [1]:
import pandas as pd
import numpy as np

n = 5000

np.random.seed(42)

data = pd.DataFrame({
    "CustomerID": np.arange(100000, 100000 + n),
    "Gender": np.random.choice(["Male", "Female"], n),
    "Age": np.random.randint(18, 70, n),
    "Tenure": np.random.randint(0, 10, n),
    "Balance": np.round(np.random.uniform(0, 250000, n), 2),
    "NumOfProducts": np.random.randint(1, 4, n),
    "HasCrCard": np.random.randint(0, 2, n),
    "IsActiveMember": np.random.randint(0, 2, n),
    "EstimatedSalary": np.round(np.random.uniform(10000, 200000, n), 2)
})

# Create churn logic (Exited column)
data["Exited"] = (
    (data["Age"] > 50) & 
    (data["IsActiveMember"] == 0) |
    (data["Balance"] > 150000) & 
    (data["NumOfProducts"] == 1)
).astype(int)

data.to_csv("customer_data.csv", index=False)
print(data.head())

   CustomerID  Gender  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0      100000    Male   68       1  210869.79              2          1   
1      100001  Female   64       5  172093.47              3          0   
2      100002    Male   64       5   73716.41              3          0   
3      100003    Male   21       9  228353.23              1          0   
4      100004    Male   24       9   68889.17              1          0   

   IsActiveMember  EstimatedSalary  Exited  
0               1        165827.37       0  
1               1        181708.70       0  
2               1         28701.68       0  
3               0         29905.13       1  
4               1        161316.00       0  


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = pd.read_csv("Customer_data.csv")

labenc = LabelEncoder()
df['Gender'] = labenc.fit_transform(df['Gender'])

x = df.drop(['CustomerID', 'Exited'], axis=1)
y = df['Exited']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)


In [3]:

#feature scaling
scaled_data = StandardScaler()
x_train = scaled_data.fit_transform(x_train)
x_test = scaled_data.transform(x_test)

#model training
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

rforest_model = RandomForestClassifier(n_estimators=100, random_state=42)
xgb_model = XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metrics='logloss', random_state=42)

rforest_model.fit(x_train, y_train)
xgb_model.fit(x_train, y_train)

rf_predictions = rforest_model.predict(x_test)
xgb_predictions = xgb_model.predict(x_test)

print("Random Forest Classifier AUC:", roc_auc_score(y_test, rforest_model.predict_proba(x_test)[:,1]))
print("XGBoost Classifier AUC:", roc_auc_score(y_test, xgb_model.predict_proba(x_test)[:,1]))

print("Random Forest Classifer Report:\n", classification_report(y_test, rf_predictions))
print("XGB Classifer Report:\n", classification_report(y_test, xgb_predictions))

C:\Users\akash\tf-env\lib\site-packages\xgboost\training.py:200: UserWarning: [13:07:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "eval_metrics", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Random Forest Classifier AUC: 1.0
XGBoost Classifier AUC: 1.0
Random Forest Classifer Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       713
           1       1.00      1.00      1.00       287

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000

XGB Classifer Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       713
           1       1.00      1.00      1.00       287

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [4]:
#Use of hyperparameter for fine tuning the model
from sklearn.model_selection import GridSearchCV

rforest_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# GridSearch
rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=rforest_param_grid,
    cv=3,
    n_jobs=-1,
    scoring='roc_auc'
)

rf_grid.fit(x_train, y_train)

print("Best RF Parameters:", rf_grid.best_params_)
print("Best RF AUC:", rf_grid.best_score_)

# Use the best estimator
best_rf = rf_grid.best_estimator_

Best RF Parameters: {'bootstrap': True, 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best RF AUC: 1.0
